# Per-Class Conformal Risk Control for Multi-Label ECG Classification

This notebook reproduces all experiments, tables, and figures for the paper
*"Per-Class Conformal Risk Control for Multi-Label ECG Classification:
Achieving Clinically-Justified False Negative Rate Guarantees"*
(El Allam & Hamlich, 2026).

## What the method does

For each diagnostic class `k`, we treat the problem as an independent binary
calibration and apply Conformal Risk Control with a Hoeffding finite-sample
correction, so that each class meets a clinically-justified false negative
rate (FNR) target with high probability (95% confidence).

## What this notebook produces

1. **Multi-label co-occurrence analysis** for PTB-XL and Chapman-Shaoxing
2. **Per-class FNR and FPR** for Standard CP vs Multi-Label CRC on both datasets
3. **Hoeffding-correction ablation** (with vs without the finite-sample correction)
4. **Calibration-set-size sensitivity** sweep
5. **Distribution-shift evaluation** (train PTB-XL -> test Chapman, harmonised labels)
6. **Pareto frontier** of FNR vs prediction-set size
7. **Subclass-level evaluation** across the 23 PTB-XL diagnostic subclasses

## Datasets (download separately)

| Dataset | Source | DOI |
|---|---|---|
| PTB-XL v1.0.3 | https://physionet.org/content/ptb-xl/1.0.3/ | 10.13026/x4td-x982 |
| Chapman-Shaoxing | https://physionet.org/content/chapman-shaoxing/1.0.1/ | 10.13026/wgex-er52 |

Runtime: ~30 min on a Colab T4 GPU, ~12 min on A100.


## 0. Setup

In [ ]:
!pip install -q wfdb scikit-learn tensorflow

import os, json, ast, glob, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from scipy.io import loadmat
from scipy.signal import resample
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/CRC_Revision'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Output directory: {OUT_DIR}")


## 1. Multi-Label Conformal Risk Control

### 1.1 What changes from single-label

In the original (single-label) formulation, the nonconformity score per sample
was a single scalar `s = 1 - p_{y_true}`, and the prediction set was built by
including every class `c` whose nonconformity `1 - p_c` was below the
class-conditional threshold `λ_c`.

In multi-label, each class becomes an **independent binary problem**:
- Nonconformity for class `k`: `s_k = 1 - p_k`, computed **only on samples where
  `y_k = 1`** (i.e. the positives for class `k`).
- The Hoeffding-corrected quantile gives a per-class threshold `λ_k`.
- Inclusion rule: class `k` is in the prediction set iff `p_k(x) ≥ 1 - λ_k`.

This is **algorithmically simpler** than the single-label version (no softmax,
no argmax fallback) and gives **per-class FNR control with high-probability
finite-sample guarantees**.

### 1.2 Hoeffding finite-sample correction — attribution

The Hoeffding-style correction used here transforms expected-value coverage
guarantees into high-probability ones, and appears in prior conformal-prediction
literature:

- Vovk, Gammerman & Shafer (2005), *Algorithmic Learning in a Random World*
- Romano, Sesia & Candès (2020), *Classification with Valid and Adaptive Coverage*, NeurIPS
- Angelopoulos & Bates (2021), *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*, arXiv:2107.07511
- Bates, Angelopoulos, Lei, Malik & Jordan (2021), *Distribution-free, Risk-controlling Prediction Sets*, JACM

Our contribution is **not** the correction itself but its application: deriving
**clinically-justified per-class α targets** for ECG classes (FDA / AHA-grounded)
and applying per-class CRC + correction to the multi-label ECG problem.

### 1.3 Definitions

For label `k ∈ {1, …, K}` and a multi-label dataset `(X_i, Y_i)` with
`Y_i ∈ {0,1}^K`:

- `FNR_k = P(k ∉ S(X) | Y_k = 1)`
- `FPR_k = P(k ∈ S(X) | Y_k = 0)`
- `Set size = E[|S(X)|]` (number of predicted labels)
- `Joint coverage = P(Y ⊆ S(X))` (all true labels captured)
- `Marginal coverage = (1/K) Σ_k P(k ∈ S(X) | Y_k = 1)`

The guarantee target is **per-class**: for each `k`, with probability `1 - δ`
(over the calibration draw), `FNR_k ≤ α_k`.


In [ ]:
class MultiLabelCRC:
    """
    Multi-label per-class Conformal Risk Control with finite-sample correction.

    Each class k is treated as an independent binary problem:
        score_k(x) = 1 - p_k(x)
    Calibrated on positives for class k: {x_i : y_{i,k} = 1}.

    Hoeffding correction (Vovk 2005, Bates 2021, Romano 2020, Angelopoulos 2021):
        alpha_effective = alpha_k - sqrt(log(1/delta) / (2 n_k))
    gives a high-probability (1 - delta) guarantee that FNR_k <= alpha_k.
    """

    def __init__(self, default_alpha: float = 0.10, confidence: float = 0.95,
                 finite_sample_correction: bool = True):
        self.default_alpha = default_alpha
        self.confidence = confidence
        self.delta = 1 - confidence
        self.finite_sample_correction = finite_sample_correction
        self.lambdas: Dict[int, float] = {}
        self.calibration_info: Dict[int, Dict] = {}

    # ---------------- Calibration ----------------

    def calibrate(self, probs_cal: np.ndarray, y_cal: np.ndarray,
                  class_alphas: Optional[Dict[int, float]] = None) -> Dict[int, float]:
        """
        Args:
            probs_cal: [n_cal, K] sigmoid probabilities
            y_cal: [n_cal, K] multi-hot true labels
            class_alphas: {k: alpha_k} per-class FNR targets
        """
        n_cal, K = probs_cal.shape
        if class_alphas is None:
            class_alphas = {k: self.default_alpha for k in range(K)}

        self.lambdas = {}
        self.calibration_info = {}

        for k in range(K):
            pos_mask = y_cal[:, k] == 1
            n_k = int(pos_mask.sum())
            alpha_k = class_alphas.get(k, self.default_alpha)

            if n_k < 2:
                self.lambdas[k] = 1.0  # no positives -> include always (vacuous)
                self.calibration_info[k] = dict(
                    n_cal=n_k, alpha_target=alpha_k, alpha_effective=alpha_k,
                    correction=0.0, threshold=1.0, vacuous=True,
                )
                continue

            # Nonconformity scores on positives
            scores_k = 1 - probs_cal[pos_mask, k]

            # Hoeffding correction
            if self.finite_sample_correction and n_k >= 10:
                correction = np.sqrt(np.log(1 / self.delta) / (2 * n_k))
                alpha_effective = max(0.005, alpha_k - correction)
            else:
                correction = 0.0
                alpha_effective = alpha_k

            # Conformal quantile (Romano et al. 2020)
            q_level = np.ceil((n_k + 1) * (1 - alpha_effective)) / n_k
            q_level = float(min(q_level, 1.0))
            lam_k = float(np.quantile(scores_k, q_level))

            self.lambdas[k] = lam_k
            self.calibration_info[k] = dict(
                n_cal=n_k, alpha_target=alpha_k, alpha_effective=alpha_effective,
                correction=correction, threshold=lam_k, vacuous=False,
            )

        return self.lambdas

    # ---------------- Prediction ----------------

    def predict(self, probs: np.ndarray) -> np.ndarray:
        """Return multi-hot prediction array [n, K]."""
        K = probs.shape[1]
        thr = np.array([1 - self.lambdas[k] for k in range(K)])  # include if p_k >= 1 - lambda_k
        return (probs >= thr[None, :]).astype(int)

    # ---------------- Metrics ----------------

    @staticmethod
    def compute_metrics(pred: np.ndarray, y_true: np.ndarray,
                        class_names: List[str]) -> Dict:
        """
        FNR_k, FPR_k, set size, joint coverage, marginal coverage.
        """
        n, K = pred.shape
        out = dict(class_fnr={}, class_fpr={}, class_tpr={}, class_set_pct={})

        for k in range(K):
            pos = y_true[:, k] == 1
            neg = y_true[:, k] == 0
            if pos.sum() > 0:
                out['class_fnr'][class_names[k]] = float(1 - pred[pos, k].mean())
                out['class_tpr'][class_names[k]] = float(pred[pos, k].mean())
            if neg.sum() > 0:
                out['class_fpr'][class_names[k]] = float(pred[neg, k].mean())
            out['class_set_pct'][class_names[k]] = float(pred[:, k].mean())

        set_sizes = pred.sum(axis=1)
        joint_covered = np.all((pred >= y_true), axis=1)  # all positives captured
        out['mean_set_size'] = float(set_sizes.mean())
        out['median_set_size'] = float(np.median(set_sizes))
        out['joint_coverage'] = float(joint_covered.mean())
        out['marginal_coverage'] = float(np.mean(list(out['class_tpr'].values())))
        return out

    @staticmethod
    def bootstrap_ci(pred: np.ndarray, y_true: np.ndarray, class_names: List[str],
                     n_boot: int = 1000, seed: int = 42) -> Dict:
        """95% bootstrap CIs on per-class FNR and FPR."""
        rng = np.random.default_rng(seed)
        n = len(y_true)
        fnr_boot = {c: [] for c in class_names}
        fpr_boot = {c: [] for c in class_names}
        for _ in range(n_boot):
            idx = rng.integers(0, n, n)
            m = MultiLabelCRC.compute_metrics(pred[idx], y_true[idx], class_names)
            for c in class_names:
                if c in m['class_fnr']:
                    fnr_boot[c].append(m['class_fnr'][c])
                if c in m['class_fpr']:
                    fpr_boot[c].append(m['class_fpr'][c])
        ci = dict(fnr={}, fpr={})
        for c in class_names:
            if fnr_boot[c]:
                ci['fnr'][c] = (float(np.percentile(fnr_boot[c], 2.5)),
                                float(np.percentile(fnr_boot[c], 97.5)))
            if fpr_boot[c]:
                ci['fpr'][c] = (float(np.percentile(fpr_boot[c], 2.5)),
                                float(np.percentile(fpr_boot[c], 97.5)))
        return ci

    def print_calibration_report(self, class_names: List[str]):
        print("\n" + "="*78)
        print("MULTI-LABEL CRC CALIBRATION REPORT (Hoeffding-corrected, 95% conf.)")
        print("="*78)
        print(f"{'Class':<10}{'n_pos':>8}{'alpha':>10}{'alpha_eff':>12}"
              f"{'correction':>12}{'lambda':>10}")
        print("-"*78)
        for k, info in self.calibration_info.items():
            print(f"{class_names[k]:<10}{info['n_cal']:>8}{info['alpha_target']:>10.1%}"
                  f"{info['alpha_effective']:>12.1%}{info['correction']:>12.4f}"
                  f"{info['threshold']:>10.4f}")
        print("="*78)


# Baselines for multi-label
def baseline_threshold(probs: np.ndarray, thr: float) -> np.ndarray:
    return (probs >= thr).astype(int)

def baseline_per_class_f1_threshold(probs_cal: np.ndarray, y_cal: np.ndarray) -> np.ndarray:
    """Tune per-class threshold to maximize F1 on calibration data."""
    K = probs_cal.shape[1]
    best_thr = np.zeros(K)
    candidates = np.linspace(0.05, 0.95, 19)
    for k in range(K):
        best_f1, best_t = -1, 0.5
        y_k = y_cal[:, k]
        if y_k.sum() == 0:
            best_thr[k] = 0.5
            continue
        for t in candidates:
            pred_k = (probs_cal[:, k] >= t).astype(int)
            tp = ((pred_k == 1) & (y_k == 1)).sum()
            fp = ((pred_k == 1) & (y_k == 0)).sum()
            fn = ((pred_k == 0) & (y_k == 1)).sum()
            if tp + fp == 0 or tp + fn == 0:
                continue
            prec = tp / (tp + fp)
            rec = tp / (tp + fn)
            f1 = 2*prec*rec / (prec + rec) if (prec + rec) > 0 else 0
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thr[k] = best_t
    return best_thr

def baseline_standard_cp_multilabel(probs_cal: np.ndarray, y_cal: np.ndarray,
                                    alpha: float = 0.10) -> np.ndarray:
    """
    Naive split-CP applied per class: single global Hoeffding-free quantile
    of nonconformity scores 1 - p_k(x_i) over positives for class k.
    """
    K = probs_cal.shape[1]
    thr = np.zeros(K)
    for k in range(K):
        pos = y_cal[:, k] == 1
        if pos.sum() < 2:
            thr[k] = 0.5
            continue
        scores = 1 - probs_cal[pos, k]
        n = pos.sum()
        q = np.ceil((n + 1) * (1 - alpha)) / n
        thr[k] = float(1 - np.quantile(scores, min(q, 1.0)))
    return thr  # interpret as: include class k if p_k >= thr[k]

print("MultiLabelCRC class and baselines loaded.")


## 2. PTB-XL Multi-Label Data Loading

### 2.1 Key change vs. original notebook
The original notebook does:
```python
Y = Y[Y.diagnostic_superclass.apply(len) == 1]  # Single label only ❌
```
We **remove** this line and instead keep all samples with at least one diagnostic
superclass label, then one-hot encode into a `[n, 5]` multi-hot matrix.

### 2.2 Split protocol
- 70% train / 15% calibration / 15% test
- Stratification: by the most-frequent positive class (multi-label stratify proxy)
- Seed: 42


In [ ]:
# Load PTB-XL metadata
import zipfile
ptb_zip = '/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip'
if not os.path.exists('/content/ptbxl'):
    with zipfile.ZipFile(ptb_zip, 'r') as z:
        z.extractall('/content/ptbxl/')

PTBXL_PATH = '/content/ptbxl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
Y = pd.read_csv(PTBXL_PATH + 'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

agg_df = pd.read_csv(PTBXL_PATH + 'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(scp_dict):
    return list(set(agg_df.loc[k, 'diagnostic_class']
                    for k in scp_dict.keys() if k in agg_df.index))

Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

# KEY CHANGE: keep samples with ≥1 label (not == 1)
Y = Y[Y.diagnostic_superclass.apply(len) >= 1]

# Multi-hot encode
PTBXL_CLASSES = sorted({c for sub in Y.diagnostic_superclass for c in sub})
print(f"PTB-XL classes: {PTBXL_CLASSES}")  # ['CD','HYP','MI','NORM','STTC']

def to_multihot(sub_list, classes):
    v = np.zeros(len(classes), dtype=np.int8)
    for c in sub_list:
        if c in classes:
            v[classes.index(c)] = 1
    return v

Y_ml = np.stack([to_multihot(s, PTBXL_CLASSES) for s in Y.diagnostic_superclass])

print(f"Samples kept (multi-label): {len(Y_ml)}")
print(f"Per-class positive counts: {dict(zip(PTBXL_CLASSES, Y_ml.sum(axis=0)))}")
print(f"Avg labels per sample: {Y_ml.sum(axis=1).mean():.2f}")
print(f"% multi-label samples (>1 positive): {(Y_ml.sum(axis=1) > 1).mean():.1%}")


In [ ]:
# Load ECG signals (100 Hz)
import wfdb
print("Loading ECG signals (this takes 3–5 min)...")
X_ptb = np.array([wfdb.rdsamp(PTBXL_PATH + row.filename_lr)[0]
                  for _, row in Y.iterrows()])
print(f"X shape: {X_ptb.shape}")  # (~21k, 1000, 12)

# Normalize per channel
mean = X_ptb.mean(axis=(0,1), keepdims=True)
std = X_ptb.std(axis=(0,1), keepdims=True) + 1e-8
X_ptb = (X_ptb - mean) / std

# Stratify proxy: most-common positive class index per sample
strat = Y_ml.argmax(axis=1)

X_tr, X_tmp, ytr, ytmp, strat_tr, strat_tmp = train_test_split(
    X_ptb, Y_ml, strat, test_size=0.30, stratify=strat, random_state=42)
X_cal, X_te, ycal, yte, _, _ = train_test_split(
    X_tmp, ytmp, strat_tmp, test_size=0.50, stratify=strat_tmp, random_state=42)

print(f"Train / Cal / Test sizes: {len(X_tr)} / {len(X_cal)} / {len(X_te)}")
print(f"Per-class positives in calibration: {dict(zip(PTBXL_CLASSES, ycal.sum(axis=0)))}")


### 2.3 Clinical FNR targets

We derive per-class FNR targets `α_k` from a hierarchy of evidence:

1. **FDA-cleared device benchmarks**:
   - Apple Watch ECG: 98.3% sensitivity (1.7% FNR) for AF (FDA De Novo DEN180044)
   - AliveCor KardiaMobile: 96.6% sensitivity for AF (FDA K181823)
   - PMcardio Queen of Hearts: 80.6% sensitivity for OMI (CE Mark)
2. **AHA / ESC sensitivity guidelines** for STEMI screening (≥95%)
3. **Diagnostic-error tolerance literature**:
   - Kelly, C.J. et al. (2019), *BMC Medicine* — Key challenges for delivering clinical impact with AI
   - Kompa, B. et al. (2021), *npj Digital Medicine* — Second opinion needed: communicating uncertainty in medical machine learning

These give us the targets:
- **MI = 5%**, **STTC = 5%** (life-threatening, time-critical)
- **CD = 10%** (moderate urgency)
- **HYP = 15%** (chronic, low-acuity)
- **NORM**: FPR-controlled, not FNR — α_NORM not enforced; we report FPR.


In [ ]:
def get_clinical_alpha_ptbxl(c):
    return {'MI': 0.05, 'STTC': 0.05, 'CD': 0.10, 'HYP': 0.15, 'NORM': 0.15}[c]

class_alphas_ptb = {i: get_clinical_alpha_ptbxl(c) for i, c in enumerate(PTBXL_CLASSES)}
print("Clinical alphas:", {c: f'{class_alphas_ptb[i]:.0%}'
                            for i, c in enumerate(PTBXL_CLASSES)})


## 3. PTB-XL Sigmoid Model + Multi-Label CRC

### 3.1 Architecture
Same backbone as the original notebook (Conv1D × 3 + GAP + Dense), with
**sigmoid output + binary cross-entropy**. This is the only model-level change
required for multi-label.


In [ ]:
def build_multilabel_cnn(input_shape, K):
    m = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(32, 7, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 5, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(K, activation='sigmoid'),  # ← key change
    ])
    m.compile(optimizer='adam',
              loss='binary_crossentropy',  # ← key change
              metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True),
                       tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='bacc')])
    return m

model_ptb = build_multilabel_cnn(X_tr.shape[1:], len(PTBXL_CLASSES))
print(f"PTB-XL multi-label model — params: {model_ptb.count_params():,}")

hist_ptb = model_ptb.fit(X_tr, ytr,
                          validation_split=0.10,
                          epochs=20, batch_size=64, verbose=2)

# Predict
probs_cal_ptb = model_ptb.predict(X_cal, verbose=0)
probs_te_ptb = model_ptb.predict(X_te, verbose=0)

# Per-class AUROC on test
from sklearn.metrics import roc_auc_score
print("\nPer-class AUROC on PTB-XL test:")
for i, c in enumerate(PTBXL_CLASSES):
    if yte[:, i].sum() > 0:
        auc = roc_auc_score(yte[:, i], probs_te_ptb[:, i])
        print(f"  {c}: AUROC = {auc:.3f}")
print(f"Macro AUROC: {roc_auc_score(yte, probs_te_ptb, average='macro'):.3f}")


### 3.2 Run multi-label CRC + baselines on PTB-XL

In [ ]:
# Multi-label CRC (ours)
crc_ptb = MultiLabelCRC(confidence=0.95, finite_sample_correction=True)
crc_ptb.calibrate(probs_cal_ptb, ycal, class_alphas_ptb)
crc_ptb.print_calibration_report(PTBXL_CLASSES)
pred_crc_ptb = crc_ptb.predict(probs_te_ptb)
m_crc_ptb = MultiLabelCRC.compute_metrics(pred_crc_ptb, yte, PTBXL_CLASSES)
ci_crc_ptb = MultiLabelCRC.bootstrap_ci(pred_crc_ptb, yte, PTBXL_CLASSES, n_boot=500)

# Baseline 1: Threshold 0.5
pred_b05 = baseline_threshold(probs_te_ptb, 0.5)
m_b05 = MultiLabelCRC.compute_metrics(pred_b05, yte, PTBXL_CLASSES)

# Baseline 2: Threshold 0.3
pred_b03 = baseline_threshold(probs_te_ptb, 0.3)
m_b03 = MultiLabelCRC.compute_metrics(pred_b03, yte, PTBXL_CLASSES)

# Baseline 3: Per-class F1-tuned
thr_f1 = baseline_per_class_f1_threshold(probs_cal_ptb, ycal)
pred_f1 = (probs_te_ptb >= thr_f1[None, :]).astype(int)
m_f1 = MultiLabelCRC.compute_metrics(pred_f1, yte, PTBXL_CLASSES)

# Baseline 4: Standard CP (multi-label, no Hoeffding correction, single alpha = 0.10)
thr_scp = baseline_standard_cp_multilabel(probs_cal_ptb, ycal, alpha=0.10)
pred_scp = (probs_te_ptb >= thr_scp[None, :]).astype(int)
m_scp = MultiLabelCRC.compute_metrics(pred_scp, yte, PTBXL_CLASSES)

def print_table(label, m, ci=None):
    print(f"\n--- {label} ---")
    print(f"{'Class':<8}{'FNR':>10}{'FPR':>10}{'Target':>10}{'Met?':>8}")
    print("-"*46)
    for c in PTBXL_CLASSES:
        fnr = m['class_fnr'].get(c, np.nan)
        fpr = m['class_fpr'].get(c, np.nan)
        idx = PTBXL_CLASSES.index(c)
        tgt = class_alphas_ptb[idx]
        met = '✓' if (not np.isnan(fnr) and fnr <= tgt) else '✗'
        if ci is not None and c in ci.get('fnr', {}):
            lo, hi = ci['fnr'][c]
            print(f"{c:<8}{fnr:>7.1%} [{lo:.1%},{hi:.1%}]  {fpr:>7.1%} {tgt:>9.0%}{met:>8}")
        else:
            print(f"{c:<8}{fnr:>10.1%}{fpr:>10.1%}{tgt:>10.0%}{met:>8}")
    print(f"Mean set size: {m['mean_set_size']:.2f}   "
          f"Joint coverage: {m['joint_coverage']:.1%}   "
          f"Marginal coverage: {m['marginal_coverage']:.1%}")

print_table("Threshold 0.5", m_b05)
print_table("Threshold 0.3", m_b03)
print_table("Per-class F1-tuned threshold", m_f1)
print_table("Standard CP (multi-label, no correction)", m_scp)
print_table("Multi-Label CRC (ours, Hoeffding-corrected)", m_crc_ptb, ci_crc_ptb)

# Targets met counter (exclude NORM since it's FPR-controlled)
fnr_classes_ptb = [c for c in PTBXL_CLASSES if c != 'NORM']
targets_met_crc = sum(m_crc_ptb['class_fnr'][c] <= class_alphas_ptb[PTBXL_CLASSES.index(c)]
                     for c in fnr_classes_ptb)
targets_met_scp = sum(m_scp['class_fnr'][c] <= class_alphas_ptb[PTBXL_CLASSES.index(c)]
                     for c in fnr_classes_ptb)
print(f"\n*** PTB-XL targets met (FNR ≤ α, excluding NORM): "
      f"CRC = {targets_met_crc}/{len(fnr_classes_ptb)}  vs  Std CP = {targets_met_scp}/{len(fnr_classes_ptb)} ***")


## 4. Chapman-Shaoxing Multi-Label Pipeline

### 4.1 Key change vs. original notebook
Original used `priority_order = [1, 4, 2, 3, 5, 0]` to collapse multi-label
SNOMED codes into a single class. We **remove** that and multi-hot encode all
mapped classes per record.


In [ ]:
# Chapman data path — adjust if needed
CHAPMAN_PATH = '/content/chapman'
data_dir = os.path.join(CHAPMAN_PATH, 'WFDB_ChapmanShaoxing')

if not os.path.exists(data_dir):
    # Use Kaggle as in original notebook
    import kaggle
    # !mkdir -p /content/chapman && cd /content/chapman && kaggle datasets download ...
    raise RuntimeError("Place Chapman-Shaoxing WFDB at /content/chapman/WFDB_ChapmanShaoxing"
                       " (reuse original notebook's Cells 23–26 if needed).")

# SNOMED → class id mapping (same as original notebook)
SNOMED_TO_CLASS = {
    '426783006': 0, '427393009': 0, '426177001': 0, '427084000': 0,
    '164934002': 0, '17621005': 0, '426761007': 0,
    '164889003': 1, '164890007': 1, '195080001': 1,
    '270492004': 2, '195042002': 2, '164909002': 2, '445118002': 2,
    '445211001': 2, '251120003': 2, '59118001': 2, '713427006': 2,
    '713426002': 2, '47665007': 2,
    '284470004': 3, '63593006': 3, '251173003': 3, '427172004': 3,
    '17338001': 3, '164884008': 3, '251180001': 3, '427665004': 3,
    '429622005': 4, '428750005': 4, '164930006': 4, '164931005': 4, '164917005': 4,
    '55827005': 5, '164873001': 5, '39732003': 5, '47567000': 5, '251146004': 5,
    '698252002': 5, '10370003': 5, '164947007': 5, '111975006': 5, '164951009': 5,
    '164861001': 5, '164865005': 5,
}
CHAPMAN_CLASSES = ['Normal', 'AF', 'CD', 'PAC_PVC', 'ST', 'Other']
CHAPMAN_ALPHAS = {'Normal': 0.15, 'AF': 0.05, 'CD': 0.10, 'PAC_PVC': 0.10,
                  'ST': 0.05, 'Other': 0.15}

hea_files = sorted(glob.glob(os.path.join(data_dir, '*.hea')))
print(f"Parsing {len(hea_files)} header files...")

records, labels_ml = [], []
for hea in hea_files:
    rec = os.path.splitext(os.path.basename(hea))[0]
    with open(hea) as f:
        content = f.read()
    m = re.search(r'#Dx:\s*(.+)', content)
    if not m:
        continue
    codes = [c.strip() for c in m.group(1).split(',')]
    mapped = sorted({SNOMED_TO_CLASS[c] for c in codes if c in SNOMED_TO_CLASS})
    if not mapped:
        continue
    # MULTI-HOT (don't collapse!)
    v = np.zeros(len(CHAPMAN_CLASSES), dtype=np.int8)
    for k in mapped:
        v[k] = 1
    records.append(rec)
    labels_ml.append(v)

Y_chap_ml = np.array(labels_ml)
print(f"Chapman records kept: {len(records)}")
print(f"Per-class positives: {dict(zip(CHAPMAN_CLASSES, Y_chap_ml.sum(axis=0)))}")
print(f"Avg labels per sample: {Y_chap_ml.sum(axis=1).mean():.2f}")
print(f"% multi-label samples: {(Y_chap_ml.sum(axis=1) > 1).mean():.1%}")


In [ ]:
# Load Chapman ECG signals
target_length = 1000
X_chap_list = []
y_chap_list = []
for rec, v in zip(records, labels_ml):
    matp = os.path.join(data_dir, rec + '.mat')
    if not os.path.exists(matp):
        continue
    try:
        ecg = loadmat(matp).get('val')
        if ecg is None:
            continue
        if ecg.shape[0] == 12:
            ecg = ecg.T
        if ecg.shape[0] != target_length:
            ecg_r = np.zeros((target_length, 12), dtype=np.float32)
            for ld in range(12):
                ecg_r[:, ld] = resample(ecg[:, ld], target_length)
            ecg = ecg_r
        X_chap_list.append(ecg.astype(np.float32))
        y_chap_list.append(v)
    except Exception:
        continue

X_chap = np.array(X_chap_list, dtype=np.float32)
Y_chap_ml = np.array(y_chap_list)
X_chap = (X_chap - X_chap.mean()) / (X_chap.std() + 1e-8)
print(f"Chapman X shape: {X_chap.shape}")

# Stratify proxy
strat_c = Y_chap_ml.argmax(axis=1)
Xtr_c, Xtmp_c, ytr_c, ytmp_c, sc_tr, sc_tmp = train_test_split(
    X_chap, Y_chap_ml, strat_c, test_size=0.30, stratify=strat_c, random_state=42)
Xcal_c, Xte_c, ycal_c, yte_c, _, _ = train_test_split(
    Xtmp_c, ytmp_c, sc_tmp, test_size=0.50, stratify=sc_tmp, random_state=42)
print(f"Chapman Train/Cal/Test: {len(Xtr_c)}/{len(Xcal_c)}/{len(Xte_c)}")

# Build & train Chapman multi-label model
model_chap = build_multilabel_cnn(Xtr_c.shape[1:], len(CHAPMAN_CLASSES))
hist_chap = model_chap.fit(Xtr_c, ytr_c, validation_split=0.10,
                            epochs=20, batch_size=64, verbose=2)
probs_cal_chap = model_chap.predict(Xcal_c, verbose=0)
probs_te_chap = model_chap.predict(Xte_c, verbose=0)

class_alphas_chap = {i: CHAPMAN_ALPHAS[c] for i, c in enumerate(CHAPMAN_CLASSES)}

# Multi-label CRC + baselines on Chapman
crc_chap = MultiLabelCRC(confidence=0.95, finite_sample_correction=True)
crc_chap.calibrate(probs_cal_chap, ycal_c, class_alphas_chap)
crc_chap.print_calibration_report(CHAPMAN_CLASSES)
pred_crc_chap = crc_chap.predict(probs_te_chap)
m_crc_chap = MultiLabelCRC.compute_metrics(pred_crc_chap, yte_c, CHAPMAN_CLASSES)
ci_crc_chap = MultiLabelCRC.bootstrap_ci(pred_crc_chap, yte_c, CHAPMAN_CLASSES, n_boot=500)

thr_scp_chap = baseline_standard_cp_multilabel(probs_cal_chap, ycal_c, alpha=0.10)
pred_scp_chap = (probs_te_chap >= thr_scp_chap[None, :]).astype(int)
m_scp_chap = MultiLabelCRC.compute_metrics(pred_scp_chap, yte_c, CHAPMAN_CLASSES)

print("\n=== Chapman Standard CP (multi-label, no correction) ===")
print_table_chap = lambda label, m, ci=None: None  # noop placeholder
# Reprint helper:
def print_chap(label, m, ci=None):
    print(f"\n--- {label} ---")
    print(f"{'Class':<10}{'FNR':>10}{'FPR':>10}{'Target':>10}{'Met?':>8}")
    print("-"*48)
    for c in CHAPMAN_CLASSES:
        fnr = m['class_fnr'].get(c, np.nan)
        fpr = m['class_fpr'].get(c, np.nan)
        tgt = CHAPMAN_ALPHAS[c]
        met = '✓' if (not np.isnan(fnr) and fnr <= tgt) else '✗'
        if ci is not None and c in ci.get('fnr', {}):
            lo, hi = ci['fnr'][c]
            print(f"{c:<10}{fnr:>7.1%} [{lo:.1%},{hi:.1%}]  {fpr:>7.1%} {tgt:>9.0%}{met:>8}")
        else:
            print(f"{c:<10}{fnr:>10.1%}{fpr:>10.1%}{tgt:>10.0%}{met:>8}")
    print(f"Mean set size: {m['mean_set_size']:.2f}  "
          f"Joint coverage: {m['joint_coverage']:.1%}  "
          f"Marginal coverage: {m['marginal_coverage']:.1%}")

print_chap("Standard CP (multi-label)", m_scp_chap)
print_chap("Multi-Label CRC (ours)", m_crc_chap, ci_crc_chap)

fnr_classes_chap = [c for c in CHAPMAN_CLASSES if c != 'Normal']
chap_met_crc = sum(m_crc_chap['class_fnr'][c] <= CHAPMAN_ALPHAS[c] for c in fnr_classes_chap)
chap_met_scp = sum(m_scp_chap['class_fnr'][c] <= CHAPMAN_ALPHAS[c] for c in fnr_classes_chap)
print(f"\n*** Chapman targets met (FNR ≤ α, excluding Normal): "
      f"CRC = {chap_met_crc}/{len(fnr_classes_chap)}  vs  Std CP = {chap_met_scp}/{len(fnr_classes_chap)} ***")

print(f"\n*** COMBINED targets met: "
      f"CRC = {targets_met_crc + chap_met_crc}/{len(fnr_classes_ptb) + len(fnr_classes_chap)}  vs  "
      f"Std CP = {targets_met_scp + chap_met_scp}/{len(fnr_classes_ptb) + len(fnr_classes_chap)} ***")


## 5. Distribution Shift Experiment

We train on PTB-XL and evaluate on Chapman-Shaoxing — a real cross-site
shift (different population, scanner, label vocabulary). Because PTB-XL and
Chapman use different label sets, we evaluate only the **harmonized labels**:

| PTB-XL class | Chapman class |
|---|---|
| MI | (no direct map — Chapman MI is in 'Other') |
| STTC | ST |
| CD | CD |
| NORM | Normal |

We map MI → 'Other' for evaluation purposes (Chapman's MI SNOMED codes were
folded into 'Other' in the original SNOMED mapping). To be safe, we restrict
the harmonized eval to the classes with direct semantic equivalents:
**{CD, STTC↔ST, NORM↔Normal}**.

**Expected finding (R2 cited Barber 2023, Ovadia 2019)**: coverage degrades
because exchangeability fails — and that is precisely the motivation for our
follow-up work on adaptive conformal inference under distribution shift
(see H-ACI, *Measurement*, in revision).


In [ ]:
# Harmonization map: PTB-XL class index -> Chapman class index (or None)
PTB_TO_CHAP = {
    PTBXL_CLASSES.index('CD'):   CHAPMAN_CLASSES.index('CD'),
    PTBXL_CLASSES.index('STTC'): CHAPMAN_CLASSES.index('ST'),
    PTBXL_CLASSES.index('NORM'): CHAPMAN_CLASSES.index('Normal'),
    # MI, HYP: no direct Chapman equivalent — excluded from harmonized eval
}
harmonized_ptb_indices = list(PTB_TO_CHAP.keys())
harmonized_chap_indices = list(PTB_TO_CHAP.values())
harmonized_names = [PTBXL_CLASSES[i] for i in harmonized_ptb_indices]

# Apply PTB-XL-trained model to Chapman data
probs_chap_via_ptb = model_ptb.predict(Xte_c, verbose=0)  # [n_chap_test, 5 PTB classes]
# Take only the harmonized columns
probs_shift = probs_chap_via_ptb[:, harmonized_ptb_indices]
y_shift = yte_c[:, harmonized_chap_indices]

# Use PTB-XL-calibrated thresholds for the harmonized classes
thr_shift = np.array([1 - crc_ptb.lambdas[i] for i in harmonized_ptb_indices])
pred_shift = (probs_shift >= thr_shift[None, :]).astype(int)
m_shift = MultiLabelCRC.compute_metrics(pred_shift, y_shift, harmonized_names)

print("=== Distribution Shift: train PTB-XL → test Chapman (harmonized labels) ===")
print(f"Harmonized class set: {harmonized_names}")
print(f"{'Class':<8}{'FNR(target?)':>14}{'FPR':>10}{'TPR':>10}")
print("-"*42)
for c in harmonized_names:
    fnr = m_shift['class_fnr'].get(c, np.nan)
    fpr = m_shift['class_fpr'].get(c, np.nan)
    tpr = m_shift['class_tpr'].get(c, np.nan)
    tgt = class_alphas_ptb[PTBXL_CLASSES.index(c)]
    print(f"{c:<8}{fnr:>8.1%} ({tgt:.0%}){fpr:>10.1%}{tpr:>10.1%}")
print(f"\nIn-distribution PTB-XL marginal coverage: {m_crc_ptb['marginal_coverage']:.1%}")
print(f"Out-of-distribution Chapman marginal coverage (harmonized): {m_shift['marginal_coverage']:.1%}")
print("→ Coverage drop quantifies exchangeability violation.")
print("→ This is the *expected and reported* failure mode (Barber et al. 2023);")
print("  follow-up work (H-ACI) introduces adaptive recalibration as mitigation.")


## 6. Pareto Frontier: FNR vs Set Size

We sweep `α` for each class jointly and plot the achieved per-class FNR and
mean set size. This quantifies the safety-vs-workload tradeoff requested by
Reviewer 2.


In [ ]:
alpha_grid = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
pareto = []
for a in alpha_grid:
    alphas_sweep = {k: a for k in range(len(PTBXL_CLASSES))}
    crc_s = MultiLabelCRC(confidence=0.95, finite_sample_correction=True)
    crc_s.calibrate(probs_cal_ptb, ycal, alphas_sweep)
    p_s = crc_s.predict(probs_te_ptb)
    m_s = MultiLabelCRC.compute_metrics(p_s, yte, PTBXL_CLASSES)
    pareto.append({'alpha': a,
                   'mean_set_size': m_s['mean_set_size'],
                   'marginal_coverage': m_s['marginal_coverage'],
                   'class_fnr': m_s['class_fnr']})
    print(f"α={a:.2f}  set_size={m_s['mean_set_size']:.2f}  "
          f"marg_cov={m_s['marginal_coverage']:.1%}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot([p['mean_set_size'] for p in pareto],
             [1 - p['marginal_coverage'] for p in pareto], 'o-')
axes[0].set_xlabel('Mean set size (clinical workload proxy)')
axes[0].set_ylabel('Marginal FNR')
axes[0].set_title('Pareto: FNR vs Set Size (PTB-XL)')
axes[0].grid(True, alpha=0.3)
for c in PTBXL_CLASSES:
    if c == 'NORM':
        continue
    axes[1].plot(alpha_grid,
                 [p['class_fnr'].get(c, np.nan) for p in pareto],
                 'o-', label=c)
axes[1].plot(alpha_grid, alpha_grid, 'k--', alpha=0.5, label='y=α (target)')
axes[1].set_xlabel('Target α')
axes[1].set_ylabel('Achieved FNR')
axes[1].set_title('Per-class FNR vs target α (PTB-XL)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'pareto_ptbxl.png'), dpi=200, bbox_inches='tight')
plt.show()


## 8. Class Co-occurrence Matrices (visual proof of multi-label)

This section makes R1's point concrete: both datasets carry substantial label
co-occurrence, so a single-label modeling choice was incorrect.


In [ ]:
def cooccurrence(Y, classes, title, save_path=None):
    '''Co-occurrence count and conditional probability matrices.'''
    K = len(classes)
    co = Y.T @ Y  # K x K count
    pos = Y.sum(axis=0)
    # Conditional: P(k | j) = co[j,k] / pos[j]
    cond = np.where(pos[:, None] > 0, co / np.maximum(pos[:, None], 1), 0.0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    im0 = axes[0].imshow(co, cmap='Blues', aspect='auto')
    axes[0].set_xticks(range(K)); axes[0].set_yticks(range(K))
    axes[0].set_xticklabels(classes, rotation=45, ha='right')
    axes[0].set_yticklabels(classes)
    axes[0].set_title(f'{title} — co-occurrence counts')
    for i in range(K):
        for j in range(K):
            axes[0].text(j, i, int(co[i, j]), ha='center', va='center',
                         color='white' if co[i, j] > co.max()/2 else 'black', fontsize=9)
    plt.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(cond, cmap='Reds', aspect='auto', vmin=0, vmax=1)
    axes[1].set_xticks(range(K)); axes[1].set_yticks(range(K))
    axes[1].set_xticklabels(classes, rotation=45, ha='right')
    axes[1].set_yticklabels(classes)
    axes[1].set_title(f'{title} — P(column | row)')
    for i in range(K):
        for j in range(K):
            axes[1].text(j, i, f'{cond[i,j]:.2f}', ha='center', va='center',
                         color='white' if cond[i, j] > 0.5 else 'black', fontsize=9)
    plt.colorbar(im1, ax=axes[1])
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    return co, cond

print("=== PTB-XL co-occurrence ===")
co_ptb, cond_ptb = cooccurrence(Y_ml, PTBXL_CLASSES,
                                'PTB-XL diagnostic superclasses',
                                save_path=os.path.join(OUT_DIR, 'cooccurrence_ptbxl.png'))
print(f"% PTB-XL samples with >1 positive label: {(Y_ml.sum(axis=1) > 1).mean():.1%}")

print("\n=== Chapman co-occurrence ===")
co_chap, cond_chap = cooccurrence(Y_chap_ml, CHAPMAN_CLASSES,
                                  'Chapman-Shaoxing',
                                  save_path=os.path.join(OUT_DIR, 'cooccurrence_chapman.png'))
print(f"% Chapman samples with >1 positive label: {(Y_chap_ml.sum(axis=1) > 1).mean():.1%}")


## 9. Macro-F1 and Per-Method Aggregate Metrics

R1 noted "Accuracy is rarely a good performance metric." We add macro-F1
and macro-AUROC across both datasets for every method.


In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

def aggregate_metrics(pred, probs, y_true):
    '''Aggregate multi-label metrics across classes.'''
    out = {}
    out['macro_f1'] = float(f1_score(y_true, pred, average='macro', zero_division=0))
    out['micro_f1'] = float(f1_score(y_true, pred, average='micro', zero_division=0))
    out['samples_f1'] = float(f1_score(y_true, pred, average='samples', zero_division=0))
    try:
        out['macro_auroc'] = float(roc_auc_score(y_true, probs, average='macro'))
        out['macro_auprc'] = float(average_precision_score(y_true, probs, average='macro'))
    except ValueError:
        out['macro_auroc'] = None
        out['macro_auprc'] = None
    return out

# PTB-XL aggregate metrics across methods
agg_ptb = {
    'Threshold 0.5':       aggregate_metrics(pred_b05, probs_te_ptb, yte),
    'Threshold 0.3':       aggregate_metrics(pred_b03, probs_te_ptb, yte),
    'Per-class F1-tuned':  aggregate_metrics(pred_f1,  probs_te_ptb, yte),
    'Standard CP':         aggregate_metrics(pred_scp, probs_te_ptb, yte),
    'Multi-Label CRC':     aggregate_metrics(pred_crc_ptb, probs_te_ptb, yte),
}
agg_chap = {
    'Standard CP':         aggregate_metrics(pred_scp_chap, probs_te_chap, yte_c),
    'Multi-Label CRC':     aggregate_metrics(pred_crc_chap, probs_te_chap, yte_c),
}

def print_agg(name, d):
    print(f"\n=== {name} aggregate metrics ===")
    print(f"{'Method':<22}{'macro F1':>10}{'micro F1':>10}{'samp F1':>10}{'AUROC':>10}{'AUPRC':>10}")
    print("-"*72)
    for meth, m in d.items():
        print(f"{meth:<22}{m['macro_f1']:>10.3f}{m['micro_f1']:>10.3f}"
              f"{m['samples_f1']:>10.3f}{(m['macro_auroc'] or 0):>10.3f}"
              f"{(m['macro_auprc'] or 0):>10.3f}")

print_agg('PTB-XL', agg_ptb)
print_agg('Chapman-Shaoxing', agg_chap)


## 10. Ablation Studies (Multi-Label)

R1 critiques the "overcorrection" of Hoeffding (FNR sitting below target → FPR
inflation). These ablations show the correction is **necessary**: without it,
FNR exceeds target ≈ half the time on small classes.

### 10.1 Hoeffding correction on vs off (PTB-XL)


In [ ]:
ablation_correction = {}
for use_corr in [True, False]:
    crc_a = MultiLabelCRC(confidence=0.95, finite_sample_correction=use_corr)
    crc_a.calibrate(probs_cal_ptb, ycal, class_alphas_ptb)
    p_a = crc_a.predict(probs_te_ptb)
    m_a = MultiLabelCRC.compute_metrics(p_a, yte, PTBXL_CLASSES)
    ablation_correction[f'correction={use_corr}'] = m_a
    print(f"\n--- correction={use_corr} ---")
    print(f"{'Class':<8}{'n_cal':>8}{'FNR':>10}{'target':>10}{'FPR':>10}{'met?':>8}")
    print("-"*54)
    for c in PTBXL_CLASSES:
        idx = PTBXL_CLASSES.index(c)
        tgt = class_alphas_ptb[idx]
        n_c = crc_a.calibration_info[idx]['n_cal']
        fnr = m_a['class_fnr'].get(c, np.nan)
        fpr = m_a['class_fpr'].get(c, np.nan)
        met = '✓' if fnr <= tgt else '✗'
        print(f"{c:<8}{n_c:>8}{fnr:>10.1%}{tgt:>10.0%}{fpr:>10.1%}{met:>8}")
    print(f"Mean set size: {m_a['mean_set_size']:.2f}")


### 10.2 Calibration-set-size sweep

We resample the calibration set at fractions of its full size and report how
FNR control degrades as `n_cal` shrinks.


In [ ]:
fractions = [0.10, 0.25, 0.50, 0.75, 1.00]
rng = np.random.default_rng(42)
ablation_calsize = []
for frac in fractions:
    n_use = int(frac * len(ycal))
    sel = rng.choice(len(ycal), size=n_use, replace=False)
    crc_s = MultiLabelCRC(confidence=0.95, finite_sample_correction=True)
    crc_s.calibrate(probs_cal_ptb[sel], ycal[sel], class_alphas_ptb)
    p_s = crc_s.predict(probs_te_ptb)
    m_s = MultiLabelCRC.compute_metrics(p_s, yte, PTBXL_CLASSES)
    ablation_calsize.append({
        'fraction': frac, 'n_cal': n_use,
        'mean_set_size': m_s['mean_set_size'],
        'marginal_coverage': m_s['marginal_coverage'],
        'class_fnr': m_s['class_fnr'],
        'class_fpr': m_s['class_fpr'],
    })
    print(f"frac={frac:.2f} n_cal={n_use:5d}  "
          f"marg_cov={m_s['marginal_coverage']:.1%}  "
          f"set_size={m_s['mean_set_size']:.2f}")

# Plot calibration-size effect on FNR for each class
fig, ax = plt.subplots(figsize=(8, 5))
for c in PTBXL_CLASSES:
    if c == 'NORM':
        continue
    vals = [a['class_fnr'].get(c, np.nan) for a in ablation_calsize]
    ax.plot([a['n_cal'] for a in ablation_calsize], vals, 'o-', label=c)
ax.set_xlabel('Calibration set size')
ax.set_ylabel('Achieved FNR (test)')
ax.set_title('Calibration set size vs FNR (PTB-XL)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig(os.path.join(OUT_DIR, 'ablation_calsize.png'), dpi=200, bbox_inches='tight')
plt.show()


## 11. Finer PTB-XL Hierarchy: Diagnostic Subclasses

R1 asked for a finer hierarchy. PTB-XL has ~23 `diagnostic_subclass` labels
(e.g. AMI, IMI, ISC_, NST_, LAFB/LPFB) within the 5 superclasses.

**What this reveals**: the Hoeffding correction `√(log(1/δ)/(2n))` requires
sufficient calibration positives per class. Below `n ≈ 30`, the correction
exceeds the target α, making the bound vacuous. We therefore report subclass
CRC for classes with `n_pos_cal ≥ 30`, and treat the rest as Future Work.


In [ ]:
# Re-aggregate at SUBCLASS level using PTB-XL metadata loaded earlier
def aggregate_subclass(scp_dict):
    return list(set(agg_df.loc[k, 'diagnostic_subclass']
                    for k in scp_dict.keys() if k in agg_df.index))

Y2 = pd.read_csv(PTBXL_PATH + 'ptbxl_database.csv', index_col='ecg_id')
Y2.scp_codes = Y2.scp_codes.apply(ast.literal_eval)
Y2['subclass'] = Y2.scp_codes.apply(aggregate_subclass)
Y2 = Y2[Y2.subclass.apply(len) >= 1]

all_subs = sorted({s for lst in Y2.subclass for s in lst})
print(f"PTB-XL subclasses ({len(all_subs)}): {all_subs}")

def to_multihot_sub(sub_list, classes):
    v = np.zeros(len(classes), dtype=np.int8)
    for s in sub_list:
        if s in classes:
            v[classes.index(s)] = 1
    return v

Y2_ml = np.stack([to_multihot_sub(s, all_subs) for s in Y2.subclass])
print(f"Subclass positives per class:")
for i, s in enumerate(all_subs):
    print(f"  {s:<10}  {Y2_ml[:, i].sum():>5}")

# Use SAME split indices as superclass run for fair comparison
# (we can't reuse the exact split since Y2 has slightly different filter, but
#  re-split with the same seed and stratify proxy)
strat2 = Y2_ml.argmax(axis=1)
# Load ECG signals only for indices we kept (subclass filter is nearly identical to superclass)
print("\nLoading signals for subclass run (≈3–5 min)...")
X_sub = np.array([wfdb.rdsamp(PTBXL_PATH + row.filename_lr)[0]
                  for _, row in Y2.iterrows()])
X_sub = (X_sub - X_sub.mean(axis=(0,1), keepdims=True)) / (X_sub.std(axis=(0,1), keepdims=True) + 1e-8)

X_tr2, X_tmp2, ytr2, ytmp2, s_tr, s_tmp = train_test_split(
    X_sub, Y2_ml, strat2, test_size=0.30, stratify=strat2, random_state=42)
X_cal2, X_te2, ycal2, yte2, _, _ = train_test_split(
    X_tmp2, ytmp2, s_tmp, test_size=0.50, stratify=s_tmp, random_state=42)

# Train subclass model
model_sub = build_multilabel_cnn(X_tr2.shape[1:], len(all_subs))
print(f"\nTraining subclass model ({len(all_subs)} subclasses)...")
model_sub.fit(X_tr2, ytr2, validation_split=0.10, epochs=20, batch_size=64, verbose=2)
probs_cal2 = model_sub.predict(X_cal2, verbose=0)
probs_te2 = model_sub.predict(X_te2, verbose=0)

# Hoeffding feasibility check: which subclasses have enough calibration positives?
delta = 0.05
HOEFFDING_FEASIBLE_MIN = 30  # arbitrary practical floor
feasibility = {}
for i, s in enumerate(all_subs):
    n_pos = int(ycal2[:, i].sum())
    if n_pos >= HOEFFDING_FEASIBLE_MIN:
        corr = np.sqrt(np.log(1/delta) / (2 * n_pos))
        feasibility[s] = {'n_pos_cal': n_pos, 'correction': float(corr), 'feasible': True}
    else:
        feasibility[s] = {'n_pos_cal': n_pos, 'correction': None, 'feasible': False}

print("\n=== Hoeffding feasibility per subclass ===")
print(f"{'Subclass':<10}{'n_pos_cal':>12}{'correction':>14}{'feasible?':>12}")
print("-"*48)
for s, info in feasibility.items():
    corr_str = f"{info['correction']:.3f}" if info['correction'] is not None else 'N/A'
    print(f"{s:<10}{info['n_pos_cal']:>12}{corr_str:>14}{str(info['feasible']):>12}")

# Run multi-label CRC at subclass level — uniform α = 0.10 on feasible subclasses only
sub_alphas = {i: 0.10 for i in range(len(all_subs))}
crc_sub = MultiLabelCRC(confidence=0.95, finite_sample_correction=True)
crc_sub.calibrate(probs_cal2, ycal2, sub_alphas)
pred_sub = crc_sub.predict(probs_te2)
m_sub = MultiLabelCRC.compute_metrics(pred_sub, yte2, all_subs)

print("\n=== Subclass CRC results (Hoeffding-corrected, α=0.10) ===")
print(f"{'Subclass':<10}{'FNR':>10}{'FPR':>10}{'TPR':>10}{'feasible?':>12}")
print("-"*52)
for i, s in enumerate(all_subs):
    fnr = m_sub['class_fnr'].get(s, np.nan)
    fpr = m_sub['class_fpr'].get(s, np.nan)
    tpr = m_sub['class_tpr'].get(s, np.nan)
    print(f"{s:<10}{fnr:>10.1%}{fpr:>10.1%}{tpr:>10.1%}{str(feasibility[s]['feasible']):>12}")

print(f"\nSubclass mean set size: {m_sub['mean_set_size']:.2f}")
print(f"Feasible subclasses (n_pos_cal ≥ {HOEFFDING_FEASIBLE_MIN}): "
      f"{sum(1 for f in feasibility.values() if f['feasible'])}/{len(all_subs)}")


## 12. Multi-Seed Reproducibility

Re-runs the CRC calibration + evaluation with 3 calibration seeds (model fixed)
to report mean ± std on every per-class FNR and FPR. Addresses reviewer
expectation of variance estimates.


In [ ]:
seeds = [42, 123, 2024]
seed_results = []
for seed in seeds:
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(ycal))  # resample calibration (model fixed)
    crc_seed = MultiLabelCRC(confidence=0.95, finite_sample_correction=True)
    crc_seed.calibrate(probs_cal_ptb[perm], ycal[perm], class_alphas_ptb)
    p_seed = crc_seed.predict(probs_te_ptb)
    m_seed = MultiLabelCRC.compute_metrics(p_seed, yte, PTBXL_CLASSES)
    seed_results.append({'seed': seed,
                          'class_fnr': m_seed['class_fnr'],
                          'class_fpr': m_seed['class_fpr'],
                          'mean_set_size': m_seed['mean_set_size'],
                          'marginal_coverage': m_seed['marginal_coverage']})

# Aggregate
def agg_mean_std(key, c):
    vals = [r[key].get(c, np.nan) for r in seed_results]
    return float(np.nanmean(vals)), float(np.nanstd(vals))

print("=== Multi-seed PTB-XL (mean ± std across 3 calibration seeds) ===")
print(f"{'Class':<8}{'FNR (mean ± std)':>22}{'FPR (mean ± std)':>22}")
print("-"*52)
for c in PTBXL_CLASSES:
    fnr_m, fnr_s = agg_mean_std('class_fnr', c)
    fpr_m, fpr_s = agg_mean_std('class_fpr', c)
    print(f"{c:<8}{fnr_m:>10.1%} ± {fnr_s:>5.1%}     {fpr_m:>10.1%} ± {fpr_s:>5.1%}")
ss_vals = [r['mean_set_size'] for r in seed_results]
cov_vals = [r['marginal_coverage'] for r in seed_results]
print(f"\nSet size: {np.mean(ss_vals):.2f} ± {np.std(ss_vals):.2f}")
print(f"Marginal coverage: {np.mean(cov_vals):.1%} ± {np.std(cov_vals):.1%}")


## 13. Publication-Ready Figures

Generates the four main figures the revised manuscript will cite.


In [ ]:
# Figure A: per-class FNR + FPR bar chart, CRC vs Standard CP
def plot_method_comparison(metrics_dict, classes, alphas, title, save_path):
    methods = list(metrics_dict.keys())
    K = len(classes)
    x = np.arange(K)
    width = 0.8 / len(methods)
    fig, (axF, axP) = plt.subplots(1, 2, figsize=(14, 5))
    for i, (m_name, m) in enumerate(metrics_dict.items()):
        fnrs = [m['class_fnr'].get(c, 0) for c in classes]
        fprs = [m['class_fpr'].get(c, 0) for c in classes]
        axF.bar(x + i*width - 0.4, fnrs, width, label=m_name)
        axP.bar(x + i*width - 0.4, fprs, width, label=m_name)
    # Target line for FNR
    for j, c in enumerate(classes):
        if isinstance(alphas, dict):
            tgt = alphas.get(c, alphas.get(j))
        else:
            tgt = alphas[j]
        if tgt is not None:
            axF.hlines(tgt, j - 0.4, j + 0.4, colors='red', linestyles='--', linewidth=1)
    axF.set_xticks(x); axF.set_xticklabels(classes, rotation=45, ha='right')
    axP.set_xticks(x); axP.set_xticklabels(classes, rotation=45, ha='right')
    axF.set_ylabel('FNR'); axP.set_ylabel('FPR')
    axF.set_title(f'{title} — Per-class FNR (red dashes = clinical target)')
    axP.set_title(f'{title} — Per-class FPR')
    axF.legend(loc='best', fontsize=8); axP.legend(loc='best', fontsize=8)
    axF.grid(True, alpha=0.3); axP.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

plot_method_comparison(
    {'Standard CP': m_scp, 'Multi-Label CRC': m_crc_ptb},
    PTBXL_CLASSES,
    {c: class_alphas_ptb[i] if c != 'NORM' else None for i, c in enumerate(PTBXL_CLASSES)},
    'PTB-XL',
    os.path.join(OUT_DIR, 'fig_ptbxl_fnr_fpr.png'))

plot_method_comparison(
    {'Standard CP': m_scp_chap, 'Multi-Label CRC': m_crc_chap},
    CHAPMAN_CLASSES,
    {c: CHAPMAN_ALPHAS[c] if c != 'Normal' else None for c in CHAPMAN_CLASSES},
    'Chapman-Shaoxing',
    os.path.join(OUT_DIR, 'fig_chapman_fnr_fpr.png'))

# Figure B: ablation — correction on/off
fig, ax = plt.subplots(figsize=(8, 5))
classes_for_ablation = [c for c in PTBXL_CLASSES if c != 'NORM']
x = np.arange(len(classes_for_ablation))
fnrs_on = [ablation_correction['correction=True']['class_fnr'].get(c, 0) for c in classes_for_ablation]
fnrs_off = [ablation_correction['correction=False']['class_fnr'].get(c, 0) for c in classes_for_ablation]
targets = [class_alphas_ptb[PTBXL_CLASSES.index(c)] for c in classes_for_ablation]
ax.bar(x - 0.2, fnrs_on, 0.4, label='With Hoeffding correction')
ax.bar(x + 0.2, fnrs_off, 0.4, label='Without correction')
for i, tgt in enumerate(targets):
    ax.hlines(tgt, i - 0.4, i + 0.4, colors='red', linestyles='--', linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(classes_for_ablation)
ax.set_ylabel('Achieved FNR')
ax.set_title('Ablation: Hoeffding correction on vs off (PTB-XL)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_ablation_correction.png'), dpi=200, bbox_inches='tight')
plt.show()

# Figure C: distribution shift comparison
fig, ax = plt.subplots(figsize=(7, 5))
classes_h = harmonized_names
x = np.arange(len(classes_h))
fnr_in = [m_crc_ptb['class_fnr'].get(c, 0) for c in classes_h]
fnr_out = [m_shift['class_fnr'].get(c, 0) for c in classes_h]
ax.bar(x - 0.2, fnr_in, 0.4, label='In-distribution (PTB-XL test)', color='steelblue')
ax.bar(x + 0.2, fnr_out, 0.4, label='Out-of-distribution (Chapman test)', color='coral')
for i, c in enumerate(classes_h):
    tgt = class_alphas_ptb[PTBXL_CLASSES.index(c)]
    ax.hlines(tgt, i - 0.4, i + 0.4, colors='red', linestyles='--', linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(classes_h)
ax.set_ylabel('FNR')
ax.set_title('Distribution shift: PTB-XL → Chapman (harmonized classes)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_distribution_shift.png'), dpi=200, bbox_inches='tight')
plt.show()

print("\nAll publication figures saved to", OUT_DIR)


## 14. Extend `results` dict with new sections (auto-appended before save)

In [ ]:
# These extra fields get bundled into `results` before the final save below
extra_results = {
    'cooccurrence': {
        'ptbxl_counts': co_ptb.tolist(),
        'ptbxl_conditional': cond_ptb.tolist(),
        'chapman_counts': co_chap.tolist(),
        'chapman_conditional': cond_chap.tolist(),
        'ptbxl_multilabel_pct': float((Y_ml.sum(axis=1) > 1).mean()),
        'chapman_multilabel_pct': float((Y_chap_ml.sum(axis=1) > 1).mean()),
    },
    'aggregate_metrics': {
        'ptbxl': agg_ptb,
        'chapman': agg_chap,
    },
    'ablations': {
        'correction_on_off': {
            'correction_on':  ablation_correction['correction=True'],
            'correction_off': ablation_correction['correction=False'],
        },
        'calibration_size_sweep': ablation_calsize,
    },
    'subclass_analysis': {
        'subclasses': all_subs,
        'feasibility': feasibility,
        'metrics': m_sub,
        'hoeffding_min_n_pos_cal': HOEFFDING_FEASIBLE_MIN,
    },
    'multi_seed': {
        'seeds': seeds,
        'per_seed': seed_results,
    },
}
print("Extra results assembled:", list(extra_results.keys()))


## 15. Save All Revision Results

Writes a single JSON containing every number the response letter needs.


In [ ]:
results = {
    'meta': {
        'notebook': 'multilabel_crc_ecg.ipynb',
        'paper': 'Per-Class Conformal Risk Control for Multi-Label ECG Classification',
        'date': time.strftime('%Y-%m-%d'),
        'revision': 'multi-label per-class CRC w/ Hoeffding correction',
    },
    'ptbxl': {
        'classes': PTBXL_CLASSES,
        'class_alphas': {c: class_alphas_ptb[i] for i, c in enumerate(PTBXL_CLASSES)},
        'n_train': int(len(X_tr)),
        'n_cal': int(len(X_cal)),
        'n_test': int(len(X_te)),
        'avg_labels_per_sample': float(Y_ml.sum(axis=1).mean()),
        'multilabel_pct': float((Y_ml.sum(axis=1) > 1).mean()),
        'macro_auroc': float(roc_auc_score(yte, probs_te_ptb, average='macro')),
        'crc_metrics': m_crc_ptb,
        'crc_bootstrap_ci': ci_crc_ptb,
        'baselines': {
            'threshold_0.5': m_b05,
            'threshold_0.3': m_b03,
            'per_class_f1_tuned': m_f1,
            'standard_cp_multilabel': m_scp,
        },
        'targets_met_crc': int(targets_met_crc),
        'targets_met_scp': int(targets_met_scp),
        'fnr_classes_count': len(fnr_classes_ptb),
    },
    'chapman': {
        'classes': CHAPMAN_CLASSES,
        'class_alphas': CHAPMAN_ALPHAS,
        'n_train': int(len(Xtr_c)),
        'n_cal': int(len(Xcal_c)),
        'n_test': int(len(Xte_c)),
        'avg_labels_per_sample': float(Y_chap_ml.sum(axis=1).mean()),
        'multilabel_pct': float((Y_chap_ml.sum(axis=1) > 1).mean()),
        'crc_metrics': m_crc_chap,
        'crc_bootstrap_ci': ci_crc_chap,
        'baselines': {
            'standard_cp_multilabel': m_scp_chap,
        },
        'targets_met_crc': int(chap_met_crc),
        'targets_met_scp': int(chap_met_scp),
        'fnr_classes_count': len(fnr_classes_chap),
    },
    'distribution_shift': {
        'protocol': 'train PTB-XL, test Chapman, harmonized labels',
        'harmonized_classes': harmonized_names,
        'metrics': m_shift,
        'in_distribution_marginal_coverage': m_crc_ptb['marginal_coverage'],
        'out_distribution_marginal_coverage': m_shift['marginal_coverage'],
    },
    'pareto': pareto,
    'combined_summary': {
        'total_fnr_targets': len(fnr_classes_ptb) + len(fnr_classes_chap),
        'crc_targets_met': int(targets_met_crc + chap_met_crc),
        'scp_targets_met': int(targets_met_scp + chap_met_scp),
    },
}

results.update(extra_results)
out_path = os.path.join(OUT_DIR, f"revision_results_{time.strftime('%Y%m%d_%H%M')}.json")
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, default=float)
print(f"✓ Saved: {out_path}")
print(f"\n=== HEADLINE NUMBERS FOR RESPONSE LETTER ===")
print(f"PTB-XL macro AUROC: {results['ptbxl']['macro_auroc']:.3f}")
print(f"PTB-XL targets met (CRC / SCP): {targets_met_crc}/{len(fnr_classes_ptb)} vs {targets_met_scp}/{len(fnr_classes_ptb)}")
print(f"Chapman targets met (CRC / SCP): {chap_met_crc}/{len(fnr_classes_chap)} vs {chap_met_scp}/{len(fnr_classes_chap)}")
print(f"Combined: {targets_met_crc + chap_met_crc}/{len(fnr_classes_ptb)+len(fnr_classes_chap)}"
      f" vs {targets_met_scp + chap_met_scp}/{len(fnr_classes_ptb)+len(fnr_classes_chap)}")
print(f"PTB-XL CRC mean set size: {m_crc_ptb['mean_set_size']:.2f}")
print(f"Chapman CRC mean set size: {m_crc_chap['mean_set_size']:.2f}")
print(f"Distribution shift coverage drop: "
      f"{m_crc_ptb['marginal_coverage']:.1%} → {m_shift['marginal_coverage']:.1%}")
